# Advanced Problems with Solutions: Comparison Operators

Topics: identity, equality, membership, ordering comparisons, mixed numeric types, chained comparisons, short-circuiting, and common traps.

## Problem 1 — Identity vs equality

Predict the result of each expression.

```python
a = [1, 2]
b = [1, 2]
c = a

a == b
a is b
a is c
a == c
```

### Solution 1

`==` compares values. `is` compares object identity.

In [1]:
a = [1, 2]
b = [1, 2]
c = a

print(a == b)  # True: same value
print(a is b)  # False: different list objects
print(a is c)  # True: c references the same object as a
print(a == c)  # True: same value

True
False
True
True


## Problem 2 — The `is None` best practice

Fix this function:

```python
def process(value):
    if value == None:
        return 'missing'
    return 'present'
```

Why is the fix better?

### Solution 2

`None` is a singleton. Use identity comparison with `None`, not equality comparison.

In [2]:
def process(value):
    if value is None:
        return 'missing'
    return 'present'


assert process(None) == 'missing'
assert process(0) == 'present'
assert process('') == 'present'

print('All tests passed.')

All tests passed.


Using `is None` avoids accidentally calling custom `__eq__` methods on user-defined objects.

## Problem 3 — Membership in lists vs dictionaries

Predict the output.

```python
data = {'a': 10, 'b': 20}

print('a' in data)
print(10 in data)
print(10 in data.values())
print(('a', 10) in data.items())
```

### Solution 3

Membership testing on a dictionary checks keys by default.

In [3]:
data = {'a': 10, 'b': 20}

print('a' in data)              # True
print(10 in data)               # False
print(10 in data.values())      # True
print(('a', 10) in data.items()) # True

True
False
True
True


## Problem 4 — List membership uses equality, not identity

Predict the output.

```python
x = [1, 2]
y = [1, 2]
items = [x]

print(y in items)
print(any(y is item for item in items))
```

### Solution 4

`in` checks equality, not identity, for list membership.

In [4]:
x = [1, 2]
y = [1, 2]
items = [x]

print(y in items)                         # True
print(any(y is item for item in items))    # False

True
False


## Problem 5 — Ordering comparison errors

Which expressions raise `TypeError`?

```python
1 < 2
1 < '2'
3 + 4j < 5 + 6j
'apple' < 'banana'
[1, 2] < [1, 3]
```

### Solution 5

Ordering must be defined between the two operands.

In [5]:
expressions = [
    "1 < 2",
    "1 < '2'",
    "3 + 4j < 5 + 6j",
    "'apple' < 'banana'",
    "[1, 2] < [1, 3]",
]

for expr in expressions:
    try:
        print(expr, '=>', eval(expr))
    except Exception as e:
        print(expr, '=>', type(e).__name__)

1 < 2 => True
1 < '2' => TypeError
3 + 4j < 5 + 6j => TypeError
'apple' < 'banana' => True
[1, 2] < [1, 3] => True


Expected result:

```text
1 < 2 => True
1 < '2' => TypeError
3 + 4j < 5 + 6j => TypeError
'apple' < 'banana' => True
[1, 2] < [1, 3] => True
```

## Problem 6 — Mixed numeric equality

Predict the result.

```python
from decimal import Decimal
from fractions import Fraction

print(1 == 1.0)
print(1 == 1 + 0j)
print(True == 1)
print(False == 0j)
print(Decimal('0.5') == Fraction(1, 2))
```

### Solution 6

Python numeric types often compare by numeric value, even if their types differ.

In [6]:
from decimal import Decimal
from fractions import Fraction

print(1 == 1.0)                         # True
print(1 == 1 + 0j)                      # True
print(True == 1)                        # True
print(False == 0j)                      # True
print(Decimal('0.5') == Fraction(1, 2)) # True

True
True
True
True
True


## Problem 7 — Chained comparisons are not repeated expressions

Predict the output.

```python
def f():
    print('calling f')
    return 10

print(1 < f() < 20)
```

### Solution 7

In a chained comparison, the middle expression is evaluated only once.

In [7]:
def f():
    print('calling f')
    return 10

print(1 < f() < 20)

calling f
True


Expected output:

```text
calling f
True
```

This behaves like:

```python
temp = f()
1 < temp and temp < 20
```

not like:

```python
1 < f() and f() < 20
```

## Problem 8 — Chained comparisons short-circuit

Predict the output.

```python
def mark(name, value):
    print(f'evaluating {name}')
    return value

print(mark('A', 5) < mark('B', 3) < mark('C', 10))
```

### Solution 8

The second comparison is skipped if the first comparison is already false.

In [8]:
def mark(name, value):
    print(f'evaluating {name}')
    return value

print(mark('A', 5) < mark('B', 3) < mark('C', 10))

evaluating A
evaluating B
False


Expected output:

```text
evaluating A
evaluating B
False
```

`C` is never evaluated.

## Problem 9 — Membership inside a chained comparison

Explain why this expression is valid and predict the result:

```python
'A' < 'a' < 'z' > 'Z' in 'ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz'
```

### Solution 9

This is a chained comparison equivalent to:

```python
'A' < 'a' and 'a' < 'z' and 'z' > 'Z' and 'Z' in 'ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz'
```

In [9]:
expr = 'A' < 'a' < 'z' > 'Z' in 'ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz'
print(expr)

True


The result is `True` because all pairwise comparisons are true.

## Problem 10 — Advanced class comparison

Create a class `Version` that supports equality and ordering by major, minor, and patch numbers.

Examples:

```python
Version(1, 2, 0) < Version(1, 2, 1)
Version(2, 0, 0) > Version(1, 9, 9)
Version(1, 0, 0) == Version(1, 0, 0)
```

### Solution 10

A clean approach is to compare tuples.

In [10]:
from functools import total_ordering

@total_ordering
class Version:
    def __init__(self, major, minor, patch):
        self.major = major
        self.minor = minor
        self.patch = patch

    def _as_tuple(self):
        return self.major, self.minor, self.patch

    def __eq__(self, other):
        if not isinstance(other, Version):
            return NotImplemented
        return self._as_tuple() == other._as_tuple()

    def __lt__(self, other):
        if not isinstance(other, Version):
            return NotImplemented
        return self._as_tuple() < other._as_tuple()

    def __repr__(self):
        return f"Version({self.major}, {self.minor}, {self.patch})"


assert Version(1, 2, 0) < Version(1, 2, 1)
assert Version(2, 0, 0) > Version(1, 9, 9)
assert Version(1, 0, 0) == Version(1, 0, 0)
assert Version(1, 0, 1) != Version(1, 0, 0)

print('All tests passed.')

All tests passed.


## Problem 11 — Find the bug: identity used for value comparison

This code sometimes appears to work:

```python
def is_zero(x):
    return x is 0
```

Fix it and explain the bug.

### Solution 11

Use `==` for value comparison. Use `is` only for identity checks, especially `is None`.

In [11]:
def is_zero(x):
    return x == 0


assert is_zero(0)
assert is_zero(0.0)
assert is_zero(False)
assert not is_zero(1)

print('All tests passed.')

All tests passed.


Note: `False == 0` is `True` because `bool` is a subclass of `int`. If you want numeric zero but not `False`, add a type check.

## Problem 12 — Robust range validation

Write a function `valid_percentage(x)` that returns `True` only when `x` is a number between `0` and `100`, inclusive.

It should return `False` for strings and complex numbers instead of raising an exception.

### Solution 12

Chained comparisons are elegant, but first guard against unsupported types.

In [12]:
from numbers import Real

def valid_percentage(x):
    return isinstance(x, Real) and 0 <= x <= 100


assert valid_percentage(0)
assert valid_percentage(50)
assert valid_percentage(100)
assert not valid_percentage(-1)
assert not valid_percentage(101)
assert not valid_percentage('50')
assert not valid_percentage(50 + 0j)

print('All tests passed.')

All tests passed.


## Summary

Best practices:

- Use `is` for identity, especially `is None`.
- Use `==` for value equality.
- Dictionary membership checks keys by default.
- `in` usually relies on equality, not identity.
- Ordering comparisons require compatible types.
- Chained comparisons evaluate middle expressions once and short-circuit.
- Prefer clear comparisons over clever ones when readability suffers.